<a href="https://colab.research.google.com/github/LeylaEynullazada/cnn-ai-image-generalization/blob/main/milestone1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/LeylaEynullazada/cnn-ai-image-generalization.git

Cloning into 'cnn-ai-image-generalization'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 54 (delta 18), reused 27 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 19.99 KiB | 10.00 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [2]:
%cd cnn-ai-image-generalization


/content/cnn-ai-image-generalization


In [3]:
!pip install -r requirements.txt

# Real vs Fake Image Classification — Milestone 1

## Project Goal
Classify images as real or AI-generated (fake) using a CNN.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q torch torchvision numpy matplotlib tqdm scikit-learn

In [ ]:
# Set data path: "." for local or Colab with data in repo. For Colab+Drive, mount and set path.
DATA_DIR = "."

Classes: ['fake', 'real']


In [ ]:
import torch
from src.dataset import get_dataloaders
from src.model import get_model
from src.train import train_full, validate
from src.evaluate import evaluate, get_metrics, plot_confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

BATCH_SIZE = 32
IMAGE_SIZE = 128
train_loader, test_loader, classes = get_dataloaders(
    data_dir=DATA_DIR, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE, num_workers=0
)
print("Classes:", classes, "| Train batches:", len(train_loader), "| Test batches:", len(test_loader))

Train:   0%|          | 0/1500 [00:00<?, ?it/s]c:\Users\leyla\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Train:   1%|          | 11/1500 [01:07<2:32:25,  6.14s/it, acc=53.4, loss=0.625]


KeyboardInterrupt: 

## Experiment 1: Baseline CNN

Train baseline CNN (5 epochs; increase for better results).

In [ ]:
model1 = get_model(num_classes=2, dropout=0.3)
history1 = train_full(
    model1, train_loader, test_loader,
    num_epochs=5, lr=1e-3, device=device,
    save_path="checkpoints/exp1_baseline.pth"
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history1["train_loss"], label="Train")
axes[0].plot(history1["val_loss"], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(history1["train_acc"], label="Train")
axes[1].plot(history1["val_acc"], label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].set_title("Accuracy")
plt.tight_layout()
plt.show()

In [ ]:
# Load best model and evaluate
model1.load_state_dict(torch.load("checkpoints/exp1_baseline.pth", map_location=device)["model_state_dict"])
y_true, y_pred, y_probs = evaluate(model1, test_loader, device)
acc, cm, report = get_metrics(y_true, y_pred, list(classes))
print("Test Accuracy:", acc)
print("Confusion Matrix:\n", cm)
print(report)
plot_confusion_matrix(cm, list(classes))

In [ ]:
show_predictions(model1, test_loader, device, list(classes), num_examples=8)

## Experiment 2: Higher Dropout

Train with dropout=0.5 to reduce overfitting.

In [ ]:
model2 = get_model(num_classes=2, dropout=0.5)
history2 = train_full(
    model2, train_loader, test_loader,
    num_epochs=5, lr=1e-3, device=device,
    save_path="checkpoints/exp2_high_dropout.pth"
)

In [ ]:
# Evaluate Experiment 2
model2.load_state_dict(torch.load("checkpoints/exp2_high_dropout.pth", map_location=device)["model_state_dict"])
y_true2, y_pred2, _ = evaluate(model2, test_loader, device)
acc2, cm2, report2 = get_metrics(y_true2, y_pred2, list(classes))
p2, r2, f1_2 = precision_recall_f1(y_true2, y_pred2)
print("Exp2 Test Accuracy:", acc2, "| Precision:", p2, "Recall:", r2, "F1:", f1_2)

## Comparison: Experiment 1 vs 2

In [ ]:
print("Comparison: Exp1 acc={:.4f} F1={:.4f} | Exp2 acc={:.4f} F1={:.4f}".format(acc, f1, acc2, f1_2))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history1["val_acc"], label="Exp1 Baseline (dropout=0.3)")
axes[0].plot(history2["val_acc"], label="Exp2 High dropout (0.5)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Val Accuracy (%)")
axes[0].legend()
axes[0].set_title("Validation Accuracy")
axes[1].plot(history1["val_loss"], label="Exp1 Baseline")
axes[1].plot(history2["val_loss"], label="Exp2 High dropout")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Val Loss")
axes[1].legend()
axes[1].set_title("Validation Loss")
plt.tight_layout()
plt.show()

## Brief Analysis

- **Experiment 1 (Baseline):** Dropout 0.3. [Fill in: validation accuracy, whether it overfits.]
- **Experiment 2 (High dropout):** Dropout 0.5. [Fill in: validation accuracy, effect on overfitting.]
- **What worked:** [e.g. augmentation helped; higher dropout reduced overfitting.]
- **What did not:** [e.g. lower learning rate slowed convergence.]